Import das Bibliotecas

In [1]:
try:
  import gdown
except ImportError:
  !pip install gdown
  import gdown
import pandas as pd
import matplotlib as plt
import seaborn as sns
import numpy as np

In [2]:
# Atribuir ID do arquivo no Google Drive
file_id = '1uofSDiBOWyqnm-tHcoeDsFvuAjgCwJsI'
dataset_url = f'https://drive.google.com/uc?id={file_id}'

# Definir nome do arquivo
output_path = 'planilhavendas.csv'

# Baixar o arquivo usando gdown
gdown.download(dataset_url, output_path, quiet=False)

# Carregar o arquivo CSV em um DataFrame, especificando o separador como ponto-e-vírgula
df = pd.read_csv(output_path,  sep=',', encoding='latin1')

#df = pd.read_csv('/content/data.csv', sep=',', encoding='latin1')

df.head()

Downloading...
From: https://drive.google.com/uc?id=1uofSDiBOWyqnm-tHcoeDsFvuAjgCwJsI
To: /content/planilhavendas.csv
100%|██████████| 45.6M/45.6M [00:00<00:00, 58.1MB/s]


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [3]:
df.shape

(541909, 8)

In [4]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


Entender o motivo da coluna InvoiceNo está como objeto

In [6]:
# Identificar valores não numéricos na coluna 'InvoiceNo'
non_numeric_invoices = df[pd.to_numeric(df['InvoiceNo'], errors='coerce').isna()]['InvoiceNo'].unique()
print(non_numeric_invoices)

# Exibir a contagem de faturas com valores não numéricos
print(f"\nNúmero de faturas com valores não numéricos: {len(non_numeric_invoices)}")

['C536379' 'C536383' 'C536391' ... 'C581499' 'C581568' 'C581569']

Número de faturas com valores não numéricos: 3839


Identifiquei que algumas invoices possuem o C que indicam que foram canceladas. Portanto vou retirar da base essas linhas.

In [7]:
# Excluindo as linhas que possuem Invoices canceladas
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Shape do DataFrame após remover faturas canceladas: {df.shape}")

Shape do DataFrame após remover faturas canceladas: (532621, 8)


Excluir linhas que possuem valores nulos de ClientID:

In [8]:
df = df.dropna()

Mudar tipo da variável da coluna CustomerID

In [9]:
# Mudar o tipo de variável do CustomerID de float para int:
df['CustomerID'] = df['CustomerID'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    397924 non-null  object 
 1   StockCode    397924 non-null  object 
 2   Description  397924 non-null  object 
 3   Quantity     397924 non-null  int64  
 4   InvoiceDate  397924 non-null  object 
 5   UnitPrice    397924 non-null  float64
 6   CustomerID   397924 non-null  int64  
 7   Country      397924 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 27.3+ MB


Ajustar coluna de data InvoiceDate:

In [10]:
# Tentando converter 'InvoiceDate' para datetime, e transformando erros em NaT
df['InvoiceDate_converted'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Verificando se há valores NaT após a conversão
problematic_dates = df[df['InvoiceDate_converted'].isna()]

if not problematic_dates.empty:
    print("Linhas com datas problemáticas (formato inválido ou ausente):")
    display(problematic_dates[['InvoiceDate', 'InvoiceDate_converted']].head())
    print(f"Total de datas problemáticas: {len(problematic_dates)}")
else:
    print("Todas as datas foram convertidas com sucesso. Não há valores problemáticos.")

Todas as datas foram convertidas com sucesso. Não há valores problemáticos.


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   InvoiceNo              397924 non-null  object        
 1   StockCode              397924 non-null  object        
 2   Description            397924 non-null  object        
 3   Quantity               397924 non-null  int64         
 4   InvoiceDate            397924 non-null  object        
 5   UnitPrice              397924 non-null  float64       
 6   CustomerID             397924 non-null  int64         
 7   Country                397924 non-null  object        
 8   InvoiceDate_converted  397924 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 30.4+ MB


In [12]:
df.to_excel('df_tratado.xlsx', index=False)

In [13]:
df.describe()

,Quantity,UnitPrice,CustomerID,InvoiceDate_converted
count,397924.000000,397924.000000,397924.000000,397924
mean,13.021823,3.116174,15294.315171,2011-07-10 23:43:36.912475648
min,1.000000,0.000000,12346.000000,2010-12-01 08:26:00
25%,2.000000,1.250000,13969.000000,2011-04-07 11:12:00
50%,6.000000,1.950000,15159.000000,2011-07-31 14:39:00
75%,12.000000,3.750000,16795.000000,2011-10-20 14:33:00
max,80995.000000,8142.750000,18287.000000,2011-12-09 12:50:00
std,180.420210,22.096788,1713.169877,NaN


### Análise RFM de Clientes

In [14]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDate_converted
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom,2010-12-01 08:26:00
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom,2010-12-01 08:26:00
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom,2010-12-01 08:26:00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom,2010-12-01 08:26:00
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom,2010-12-01 08:26:00
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850,United Kingdom,2010-12-01 08:26:00
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850,United Kingdom,2010-12-01 08:26:00
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850,United Kingdom,2010-12-01 08:28:00
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850,United Kingdom,2010-12-01 08:28:00
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047,United Kingdom,2010-12-01 08:34:00


In [15]:
# Construir um df_RFM para agrupar por CustomerID as informações necessárias:

df['ValorTotal'] = df['Quantity'] * df['UnitPrice']

df_RFM = df.groupby('CustomerID').agg(
    UltimaCompra = ('InvoiceDate_converted', 'max'),
    Frequencia = ('InvoiceNo', 'nunique'),
    Monetario = ('ValorTotal', 'sum')
).reset_index()

df_RFM.head()

,CustomerID,UltimaCompra,Frequencia,Monetario
0,12346,2011-01-18 10:01:00,1,77183.60
1,12347,2011-12-07 15:52:00,7,4310.00
2,12348,2011-09-25 13:13:00,4,1797.24
3,12349,2011-11-21 09:51:00,1,1757.55
4,12350,2011-02-02 16:01:00,1,334.40


In [16]:
data_estudo = pd.to_datetime('12-30-2011')
data_estudo

Timestamp('2011-12-30 00:00:00')

In [17]:
df_RFM['Recencia'] = (data_estudo - df_RFM['UltimaCompra']).dt.days
df_RFM.head()

,CustomerID,UltimaCompra,Frequencia,Monetario,Recencia
0,12346,2011-01-18 10:01:00,1,77183.60,345
1,12347,2011-12-07 15:52:00,7,4310.00,22
2,12348,2011-09-25 13:13:00,4,1797.24,95
3,12349,2011-11-21 09:51:00,1,1757.55,38
4,12350,2011-02-02 16:01:00,1,334.40,330


In [18]:
df_RFM = df_RFM.drop(columns=['UltimaCompra'])
df_RFM.head()

,CustomerID,Frequencia,Monetario,Recencia
0,12346,1,77183.60,345
1,12347,7,4310.00,22
2,12348,4,1797.24,95
3,12349,1,1757.55,38
4,12350,1,334.40,330


In [19]:
# Criando colunas com scores para os valores de R F e M:
df_RFM['R'] = pd.qcut(df_RFM['Recencia'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
df_RFM['F'] = pd.qcut(df_RFM['Frequencia'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
df_RFM['M'] = pd.qcut(df_RFM['Monetario'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

#Concatenando os Scores:
df_RFM['RFM_Score'] = df_RFM['R'].astype(str) + df_RFM['F'].astype(str) + df_RFM['M'].astype(str)

df_RFM

,CustomerID,Frequencia,Monetario,Recencia,R,F,M,RFM_Score
0,12346,1,77183.60,345,1,1,5,115
1,12347,7,4310.00,22,5,5,5,555
2,12348,4,1797.24,95,2,4,4,244
3,12349,1,1757.55,38,4,1,4,414
4,12350,1,334.40,330,1,1,2,112
...,...,...,...,...,...,...,...,...
4334,18280,1,180.60,297,1,2,1,121
4335,18281,1,80.82,200,1,2,1,121
4336,18282,2,178.05,27,5,3,1,531
4337,18283,16,2094.88,23,5,5,5,555


In [20]:
# Categorizando os clientes conforme pontuação RFM:
segmentos_regras = {
    # R=4 ou 5 e F=3, 4 ou 5 (Comportamento excelente e ativo)
    r'[4-5][3-5][1-5]': '1. VIPs',
    # R=3, 4 ou 5 e F=1 ou 2 (Compraram recentemente, mas ainda poucas vezes)
    r'[3-5][1-2][1-5]': '2. Potenciais / Novos',
    # R=2 ou 3 e F=3, 4 ou 5 (Clientes frequentes que começaram a sumir) OR R=2 e F=1-2
    r'[2-3][1-5][1-5]': '3. Em Risco',
    # R=1 (Sumidos há muito tempo, independentemente de quantas vezes compraram)
    r'1[1-5][1-5]': '4. Inativos / Perdidos'
}

df_RFM['Segmento'] = df_RFM['RFM_Score'].replace(segmentos_regras, regex=True)

df_RFM

,CustomerID,Frequencia,Monetario,Recencia,R,F,M,RFM_Score,Segmento
0,12346,1,77183.60,345,1,1,5,115,4. Inativos / Perdidos
1,12347,7,4310.00,22,5,5,5,555,1. VIPs
2,12348,4,1797.24,95,2,4,4,244,3. Em Risco
3,12349,1,1757.55,38,4,1,4,414,2. Potenciais / Novos
4,12350,1,334.40,330,1,1,2,112,4. Inativos / Perdidos
...,...,...,...,...,...,...,...,...,...
4334,18280,1,180.60,297,1,2,1,121,4. Inativos / Perdidos
4335,18281,1,80.82,200,1,2,1,121,4. Inativos / Perdidos
4336,18282,2,178.05,27,5,3,1,531,1. VIPs
4337,18283,16,2094.88,23,5,5,5,555,1. VIPs


### Análise ABC Clientes

In [21]:
valortotal = df_RFM['Monetario'].sum()
valortotal

np.float64(8911407.904000001)

In [22]:
#ordenar pela coluna Monetario:
df_RFM = df_RFM.sort_values('Monetario', ascending=False).reset_index(drop=True)

#coluna com soma acumulada após ordenar:
df_RFM['Monetario_Acumulado'] = df_RFM['Monetario'].cumsum()

#calcular em porcentagem o valor acumulado em relação ao valortotal
df_RFM['Porcentagem_Acumulada'] = 100 * (df_RFM['Monetario_Acumulado'] / valortotal)

df_RFM

,CustomerID,Frequencia,Monetario,Recencia,R,F,M,RFM_Score,Segmento,Monetario_Acumulado,Porcentagem_Acumulada
0,14646,74,280206.02,21,5,5,5,555,1. VIPs,280206.020,3.144352
1,18102,60,259657.30,20,5,5,5,555,1. VIPs,539863.320,6.058115
2,17450,46,194550.79,28,5,5,5,555,1. VIPs,734414.110,8.241280
3,16446,2,168472.50,20,5,3,5,535,1. VIPs,902886.610,10.131807
4,14911,201,143825.06,21,5,5,5,555,1. VIPs,1046711.670,11.745750
...,...,...,...,...,...,...,...,...,...,...,...
4334,17956,1,12.75,269,1,2,1,121,4. Inativos / Perdidos,8911391.054,99.999811
4335,16454,2,6.90,64,3,3,1,331,3. Em Risco,8911397.954,99.999888
4336,14792,1,6.20,83,3,1,1,311,2. Potenciais / Novos,8911404.154,99.999958
4337,16738,1,3.75,317,1,2,1,121,4. Inativos / Perdidos,8911407.904,100.000000


In [23]:
#função para classificar ABC:
def classe_ABC (porcentagem):
  if porcentagem <= 80:
    return 'A'
  elif porcentagem <= 95:
    return 'B'
  else:
    return 'C'

df_RFM['Classe_ABC'] = df_RFM['Porcentagem_Acumulada'].apply(classe_ABC)

df_RFM

,CustomerID,Frequencia,Monetario,Recencia,R,F,M,RFM_Score,Segmento,Monetario_Acumulado,Porcentagem_Acumulada,Classe_ABC
0,14646,74,280206.02,21,5,5,5,555,1. VIPs,280206.020,3.144352,A
1,18102,60,259657.30,20,5,5,5,555,1. VIPs,539863.320,6.058115,A
2,17450,46,194550.79,28,5,5,5,555,1. VIPs,734414.110,8.241280,A
3,16446,2,168472.50,20,5,3,5,535,1. VIPs,902886.610,10.131807,A
4,14911,201,143825.06,21,5,5,5,555,1. VIPs,1046711.670,11.745750,A
...,...,...,...,...,...,...,...,...,...,...,...,...
4334,17956,1,12.75,269,1,2,1,121,4. Inativos / Perdidos,8911391.054,99.999811,C
4335,16454,2,6.90,64,3,3,1,331,3. Em Risco,8911397.954,99.999888,C
4336,14792,1,6.20,83,3,1,1,311,2. Potenciais / Novos,8911404.154,99.999958,C
4337,16738,1,3.75,317,1,2,1,121,4. Inativos / Perdidos,8911407.904,100.000000,C


### Análise ABC Produtos

In [24]:
print(df['StockCode'].value_counts())

StockCode
85123A    2035
22423     1724
85099B    1618
84879     1408
47566     1397
          ... 
90214Z       1
90214U       1
23449        1
90214W       1
23843        1
Name: count, Length: 3665, dtype: int64


In [25]:
df_ABC_produtos = df.groupby('StockCode').agg(
    Description = ('Description', 'first'),
    ValorTotal = ('ValorTotal', 'sum')
).reset_index()

df_ABC_produtos

,StockCode,Description,ValorTotal
0,10002,INFLATABLE POLITICAL GLOBE,699.550
1,10080,GROOVY CACTUS INFLATABLE,114.410
2,10120,DOGGY RUBBER,40.530
3,10123C,HEARTS WRAPPING TAPE,3.250
4,10124A,SPOTS ON RED BOOKCOVER TAPE,6.720
...,...,...,...
3660,C2,CARRIAGE,6686.000
3661,DOT,DOTCOM POSTAGE,11906.360
3662,M,Manual,53779.930
3663,PADS,PADS TO MATCH ALL CUSHIONS,0.003


In [26]:
#Ordenar por valor total:
df_ABC_produtos = df_ABC_produtos.sort_values('ValorTotal', ascending=False).reset_index(drop=True)

#Variável que recebe o valor total dos produtos:
valortotalprodutos = df_ABC_produtos['ValorTotal'].sum()

#Cria coluna com valor total acumalado
df_ABC_produtos['ValorTotal_Acumulado'] = df_ABC_produtos['ValorTotal'].cumsum()

#Calcula a porcetagem acumulada para cada produto
df_ABC_produtos['Porcentagem_Acumulada'] = 100 * (df_ABC_produtos['ValorTotal_Acumulado'] / valortotalprodutos)

df_ABC_produtos

,StockCode,Description,ValorTotal,ValorTotal_Acumulado,Porcentagem_Acumulada
0,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.600,168469.600,1.890494
1,22423,REGENCY CAKESTAND 3 TIER,142592.950,311062.550,3.490611
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100603.500,411666.050,4.619540
3,85099B,JUMBO BAG RED RETROSPOT,85220.780,496886.830,5.575851
4,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.730,578303.560,6.489475
...,...,...,...,...,...
3660,90084,PINK CRYSTAL GUITAR PHONE CHARM,0.850,8911405.791,99.999976
3661,90104,PURPLE FRANGIPANI HAIRCLIP,0.850,8911406.641,99.999986
3662,21268,VINTAGE BLUE TINSEL REEL,0.840,8911407.481,99.999995
3663,84227,HEN HOUSE W CHICK IN NEST,0.420,8911407.901,100.000000


In [27]:
#Aplicando a função já para classificar em ABC:
df_ABC_produtos['Classe_ABC'] = df_ABC_produtos['Porcentagem_Acumulada'].apply(classe_ABC)

df_ABC_produtos

,StockCode,Description,ValorTotal,ValorTotal_Acumulado,Porcentagem_Acumulada,Classe_ABC
0,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.600,168469.600,1.890494,A
1,22423,REGENCY CAKESTAND 3 TIER,142592.950,311062.550,3.490611,A
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100603.500,411666.050,4.619540,A
3,85099B,JUMBO BAG RED RETROSPOT,85220.780,496886.830,5.575851,A
4,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.730,578303.560,6.489475,A
...,...,...,...,...,...,...
3660,90084,PINK CRYSTAL GUITAR PHONE CHARM,0.850,8911405.791,99.999976,C
3661,90104,PURPLE FRANGIPANI HAIRCLIP,0.850,8911406.641,99.999986,C
3662,21268,VINTAGE BLUE TINSEL REEL,0.840,8911407.481,99.999995,C
3663,84227,HEN HOUSE W CHICK IN NEST,0.420,8911407.901,100.000000,C


### Salvar os arquivos em Excel (.xlsx) para serem usados no PowerBI

In [28]:
df_RFM.to_excel('df_RFM.xlsx', index=False)
df_ABC_produtos.to_excel('df_ABC_produtos.xlsx', index=False)